In [19]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms
import torch.optim as optim

In [20]:
device = "cuda" if torch.cuda.is_available() else "gpu"
device

'cuda'

In [21]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=(0.4914, 0.4822, 0.4465),
        std=(0.2470, 0.2435, 0.2616)
    )
])

In [22]:
train_datasets = datasets.CIFAR10(
    root='./data',
    train=True,
    download=True,
    transform=transform
)

test_datasets = datasets.CIFAR10(
    root='./data',
    train=False,
    download=True,
    transform=transform
)

c:\Users\oguzh\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [23]:
train_datasets, val_datasets = random_split(train_datasets, lengths= [0.85,0.15])

len(train_datasets), len(val_datasets)

(42500, 7500)

In [24]:
train_dataloader = DataLoader(
    dataset= train_datasets,
    batch_size= 32,
    shuffle = True,
    num_workers= 4,
    pin_memory= True,
)

val_dataloader = DataLoader(
    dataset= val_datasets,
    batch_size= 32,
    shuffle= False,
    num_workers= 4,
    pin_memory= True
)

test_dataloader = DataLoader(
    dataset= test_datasets,
    batch_size= 32,
    shuffle = False,
    num_workers= 4,
    pin_memory= True
)

In [25]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction= 4):
        super().__init__()
        self.pool = nn.AdaptiveAvgPool2d((1,1))

        hidden_channels = channels // reduction

        self.layer = nn.Sequential(
            nn.Conv2d(in_channels= channels, out_channels= hidden_channels, kernel_size= 1),
            nn.ReLU(),
            nn.Conv2d(in_channels= hidden_channels, out_channels= channels, kernel_size= 1),
            nn.Hardsigmoid()
        )
    
    def forward(self, X):
        scale = self.pool(X)
        scale = self.layer(X)

        return X * scale


In [26]:
class MobileNetV3Block(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        expansion,
        kernel_size,
        stride,
        use_se=True
    ):
        super().__init__()

        hidden_channels = in_channels * expansion

        # Expansion
        self.expand = nn.Sequential(
            nn.Conv2d(
                in_channels,
                hidden_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(hidden_channels),
            nn.Hardswish()
        )

        # Depthwise
        self.depthwise = nn.Sequential(
            nn.Conv2d(
                hidden_channels,
                hidden_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=kernel_size // 2,
                groups=hidden_channels,
                bias=False
            ),
            nn.BatchNorm2d(hidden_channels),
            nn.Hardswish()
        )

        # SE
        self.se = SEBlock(hidden_channels) if use_se else nn.Identity()

        # Projection
        self.project = nn.Sequential(
            nn.Conv2d(
                hidden_channels,
                out_channels,
                kernel_size=1,
                bias=False
            ),
            nn.BatchNorm2d(out_channels)
        )

        # Residual şartları
        self.use_residual = (
            stride == 1 and
            in_channels == out_channels
        )

    def forward(self, x):

        identity = x

        x = self.expand(x)
        x = self.depthwise(x)
        x = self.se(x)
        x = self.project(x)

        if self.use_residual:
            x = x + identity

        return x